# Stage 12B: learned 61-state emission TCN

Standalone Colab notebook. It trains five OOF candidate-shared TCNs on the fixed `w075_cap50` surface and does not create a Kaggle submission. Use a T4 GPU and High-RAM when available.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import importlib.util, json, os, shutil, subprocess, sys

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
required = ['torch', 'pandas', 'pyarrow', 'sklearn', 'yaml']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyarrow', 'scikit-learn', 'pyyaml'], check=True)
import torch
assert torch.cuda.is_available(), 'Change the Colab runtime to a T4 GPU and reconnect.'
artifact_dir.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Locate prerequisite OOF artifacts

The expensive Stage 11, 11C, and 12A results are reused from Drive. If one is absent, this notebook rebuilds it automatically, so a fresh Colab session can start here.


In [ ]:
STAGE11_RUN_ID = 'stage11_multicut_delta_u_full_v001'
STAGE11C_RUN_ID = 'stage11c_delta_u_robust_gate_full_v001'
STAGE12A_RUN_ID = 'stage12a_raw_ncc_benchmark_full_v001'
stage11_run = artifact_dir / STAGE11_RUN_ID
stage11c_run = artifact_dir / STAGE11C_RUN_ID
stage12a_run = artifact_dir / STAGE12A_RUN_ID
if not (stage11_run / 'fold_coefficient_oof.parquet').is_file():
    subprocess.run(['uv', 'run', 'rogii-delta-u', '--config', 'configs/experiment/stage11_multicut_delta_u.yaml', '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', STAGE11_RUN_ID], cwd=repo_dir, check=True)
if not (stage11c_run / 'gate_summary.json').is_file():
    subprocess.run(['uv', 'run', 'rogii-delta-u-gate', '--config', 'configs/experiment/stage11c_delta_u_robust_gate.yaml', '--stage11-run', str(stage11_run), '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', STAGE11C_RUN_ID], cwd=repo_dir, check=True)
if not (stage12a_run / 'benchmark_summary.json').is_file():
    subprocess.run(['uv', 'run', 'rogii-raw-ncc', '--config', 'configs/experiment/stage12a_raw_ncc_benchmark.yaml', '--stage11-run', str(stage11_run), '--stage11c-run', str(stage11c_run), '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', STAGE12A_RUN_ID], cwd=repo_dir, check=True)
assert json.loads((stage11c_run / 'gate_summary.json').read_text())['promoted_to_stage12'] is True
assert json.loads((stage12a_run / 'benchmark_summary.json').read_text())['promoted_to_learned_emission'] is True
print('Prerequisites verified:', STAGE11_RUN_ID, STAGE11C_RUN_ID, STAGE12A_RUN_ID)


## Train five OOF emission models

`LIMIT_WELLS = 24` is a wiring test only. Keep `None` for the decision run. Cost volumes are built in RAM, while checkpoints, logs, OOF predictions, and the gate summary are saved to Drive. A T4 is sufficient; dual-GPU is not used.


In [ ]:
RUN_ID = 'stage12b_learned_emission_tcn_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        sys.executable, '-m', 'rogii.cli.emission_tcn',
        '--config', 'configs/experiment/stage12b_learned_emission_tcn.yaml',
        '--stage11-run', str(stage11_run), '--stage11c-run', str(stage11c_run),
        '--stage12a-run', str(stage12a_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID, '--device', 'cuda', '--resume',
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    emission_env = os.environ.copy()
    emission_env['PYTHONPATH'] = str(repo_dir / 'src') + ':' + emission_env.get('PYTHONPATH', '')
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(command, cwd=repo_dir, env=emission_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end=''); log_handle.write(line); log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        print('\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-160:]))
        raise RuntimeError(f'Stage 12B failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


## Decision

Promotion requires learned Top-5/10 and NLL to beat the frozen raw-NCC distribution with consistent OOF gains. Expected-offset and MAP RMSE are diagnostics only; path decoding is deliberately deferred to Stage 12C.


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text(encoding='utf-8'))
{key: summary[key] for key in [
    'promoted_to_spatial_emission_audit', 'raw', 'learned', 'top10_gain', 'top5_gain',
    'nll_delta', 'surface_rmse', 'expected_offset_rmse', 'expected_offset_delta',
    'map_offset_rmse', 'map_offset_delta', 'improved_folds', 'fold_report',
    'spatial_group_top10', 'typewell_group_top10', 'gates', 'next_step',
]}


## Send back

Send the complete dictionary from the previous cell. Do not submit anything to Kaggle from this notebook.
